In [1]:
import sys
from pathlib import Path
import json
from datasets import load_dataset
import torch
import torch.nn as nn

In [2]:
# Thư mục gốc của project
ROOT_DIR = Path.cwd().parent
ROOT_DIR

WindowsPath('D:/Code Module 6/VietNamese_Image_Captioning_with_CNN_LTSM')

In [3]:
dataset = load_dataset("ai-enthusiasm-community/KTVIC")

In [4]:
dataset['train'][0]

{'image_uid': '000000000001',
 'caption_uid': ['000000000001_1',
  '000000000001_2',
  '000000000001_3',
  '000000000001_4',
  '000000000001_5'],
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=695x456>,
 'caption_vi': ['ba chiếc thuyền đang di chuyển ở trên con sông',
  'có ba con thuyền đang di chuyển trên con sông',
  'trên dòng sông có ba con thuyền đang di chuyển',
  'ba con thuyền đang di chuyển bên một cánh đồng lúa',
  'ba chiếc thuyền đang chuyển động trên một con sông'],
 'segment_caption_vi': ['ba chiếc thuyền đang di_chuyển ở trên con sông',
  'có ba con thuyền đang di_chuyển trên con sông',
  'trên dòng sông có ba con thuyền đang di_chuyển',
  'ba con thuyền đang di_chuyển bên một cánh đồng lúa',
  'ba chiếc thuyền đang chuyển_động trên một con sông']}

In [5]:
SOURCE_DIR = ROOT_DIR/'source'
BASEMODEL_DIR = ROOT_DIR/'base_model'
CONFIG_DIR = ROOT_DIR/'configuration'

In [6]:
sys.path.extend([str(SOURCE_DIR), str(SOURCE_DIR), str(BASEMODEL_DIR), str(CONFIG_DIR)])

In [7]:
from ResNet_model import *

In [8]:
class get_output_LSTM(nn.Module):
    def __init__(self):
        super().__init__()
    def forword(self,X):
        output, _ = X
        return output

In [ ]:
class GenerateCaptionImage(ResNet):
    def __init__(self, versionCNN = 34, num_classes = 512, hidden_size_LSTM = 150 , embedd_dim = 150, num_layer_LSTM = 1, vocab_size = 500, vocab =None):
        super().__init__()
        self.num_classes = num_classes
        self.version = versionCNN
        self.model = self.model[:-1]
        self.embedd_dim = embedd_dim
        self.hidden_size_LSTM = self.hidden_size_LSTM
        self._init_cell_state = nn.Linear(in_features=self.list_channel[-1], out_features=self.hidden_size_LSTM)
        self._init_hidden_state = nn.Linear(in_features=self.list_channel[-1], out_features=self.hidden_size_LSTM)
        self.num_layer_LSTM = num_layer_LSTM
        self.vocab_size = vocab_size
        self.vocab = vocab
        self.embedding_layer = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim= self.embedd_dim,
        )
        self.rnn = nn.LSTM(
            input_size = self.embedd_dim,
            hidden_size = self.hidden_size_LSTM,
            batch_first = True
        )
        self.Linear_Layer = nn.Linear(in_features=self.hidden_size_LSTM, out_features=self.vocab_size)
    def forward(self,X, captions):
        if self.model is not None:
            feature_maps = self.model(X)
        # Cell_state0
        c0 = self._init_cell_state(feature_maps).unsqueeze(0) # input of h0, c0 in LSTM has shape (1*num_layer, batch_size, hidden_size)
        h0 = self._init_hidden_state(feature_maps).unsqueeze(0)

        c0 = c0.repeat(self.num_layer, 1, 1)
        h0 = c0.repeat(self.num_layer, 1, 1)

        caption_embedded = self.embedding_layer(captions)

        output, _ = self.rnn(caption_embedded, (h0, c0))

        return self.Linear_Layer(output)
    def get_accuracy(self, logits, y):
        return torch.mean((torch.argmax(logits, dim = 2 ) ==y).float())
    def predict(self,X):
        with torch.no_grad():
            self.model.eval()
            X = X.to(self.device)
            sentence = ['<start>']
            while '<end>' not in sentence:
                logits = self.forward(X,torch.tensor(self.vocab(sentence)).squeeze(0))
                idx_token = torch.argmax(logits, dim =2)
                sentence += [self.vocab.idx2str[idx_token]]
        caption = " ".join(sentence)